# From raw/nis/20* we will extract data from our .DAT file to generate CSV's and merge into one

### Let's do a quick demo before building the function: ```read_nis```
Using
- raw/nis/2015/NIS-PUF15.SAS
- raw/nis/2015/NISPUF15.DAT

In [1]:
with open('../raw/nis/2015/NIS-PUF15.SAS', "r", encoding="latin1") as f:
        sas_text = f.read()

### We need to parse value and their numerical reps
Example
```
value SEX
. = "MISSING"
1 = "MALE"
2 = "FEMALE"
77 = "DON'T KNOW"
99 = "REFUSED"
```

### Read the .DAT file

In [2]:
import pandas as pd

df = pd.read_fwf(
    "../raw/nis/2015/NISPUF15.DAT",
    colspecs=[(176, 177)],
    names=["SEX"]
)

sex_map = {
    1: "MALE",
    2: "FEMALE",
    77: "DON'T KNOW",
    99: "REFUSED",
}

df["SEX_label"] = df["SEX"].map(sex_map)

df.head()

,SEX,SEX_label
0,1,MALE
1,1,MALE
2,1,MALE
3,2,FEMALE
4,2,FEMALE


### According to ChatGPT, these are the most important columns for vaccine coverage:
| Column         | Level          | What it means                | Why you need it                                                            |
| -------------- | -------------- | ---------------------------- | -------------------------------------------------------------------------- |
| **`SEQNUMC`**  | Child          | Unique child identifier      | Identifies the individual child/record                                     |
| **`SEQNUMHH`** | Household      | Household identifier         | Identifies which household the child belongs to; used in survey design     |
| **`STRATUM`**  | Sampling group | Survey sampling stratum      | Identifies the sampling group; needed for correct SEs/CIs                  |
| **`PROVWT_D`** | Child          | Provider-phase survey weight | Determines how much the child contributes to population estimates          |
| **`P_UTDMCV`** | Child          | Measles vaccination status   | `1` = received ≥1 qualifying measles-containing vaccination; `0` = did not |
| **`STATE`**    | Geography      | State code                   | Lets you calculate vaccination coverage by state                           |
| **`YEAR`**     | Time           | Survey year                  | Lets you calculate and compare coverage over time                          |



In [3]:
import pandas as pd

colspecs = [
    (6, 11),      # SEQNUMHH
    (12, 32),     # PROVWT_D
    (90, 94),     # STRATUM
    (94, 98),     # YEAR
    (182, 184),   # STATE
    (253, 254),   # P_UTDMCV
]

names = [
    "SEQNUMHH",
    "PROVWT_D",
    "STRATUM",
    "YEAR",
    "STATE",
    "P_UTDMCV",
]

df = pd.read_fwf(
    "../raw/nis/2015/NISPUF15.DAT",
    colspecs=colspecs,
    names=names,
    na_values=["."]
)

df.head()

,SEQNUMHH,PROVWT_D,STRATUM,YEAR,STATE,P_UTDMCV
0,1,77.861750,2017,2015,42,1.0
1,2,NaN,2072,2015,15,NaN
2,3,73.609547,2019,2015,54,1.0
3,4,NaN,2002,2015,25,NaN
4,5,141.333362,2075,2015,16,1.0


In [4]:
len(df)

27592

In [5]:
df.dtypes

SEQNUMHH      int64
PROVWT_D    float64
STRATUM       int64
YEAR          int64
STATE         int64
P_UTDMCV    float64
dtype: object

## Let's start working on this function comprised of four steps:
1. get column row from .SAS
2. parse data and turn to df
3. translate STATE
4. save df as csv

In [36]:
import us

def translate_state(sas_text):

    # Get STATE value block
    match = re.search(
        r"value\s+STATE\b(.*?);",
        sas_text,
        flags=re.IGNORECASE | re.DOTALL
    )

    state_block = match.group(1)

    # Extract number = "STATE NAME"
    pairs = re.findall(
        r'(\d+)\s*=\s*"([^"]+)"',
        state_block
    )

    # these are exceptions since the US library doesn't pick up on them
    name_fixes = {
        "DISTRICT OF COLUMBIA": "DC",
        "U.S.  VIRGIN ISLANDS": "VI",
    }

    state_map = {}

    for code, state in pairs:

        if state in name_fixes:
            state_map[int(code)] = name_fixes[state]
            continue

        result = us.states.lookup(state)

        if result:
            state_map[int(code)] = result.abbr
        else:
            print(f"Not found: {code} = {state}")

    return state_map

In [37]:
import re

def parse_dat_to_csv(sas_file_path, dat_file_path, csv_output_path):

    # define the core column names
    names = [
        "SEQNUMHH",
        "PROVWT_D",
        "STRATUM",
        "YEAR",
        "STATE",
        "P_UTDMCV",
    ]

    # parse col locations and save in array
    colspecs = []
    with open(sas_file_path, "r", encoding="latin1") as f:
        sas_text = f.read()


        for name in names:
            match = re.search(
                rf"@\d+\s+{re.escape(name)}\s+\$?\d+(?:\.\d*)?",
                sas_text
            )

            col_loc = match.group()

            numbers = re.findall(r"\d+", col_loc)

            start = int(numbers[0]) - 1
            end = start + int(numbers[1])

            colspecs.append((start, end))

    df = pd.read_fwf(
        dat_file_path,
        colspecs=colspecs,
        names=names,
        na_values=["."]
    )

    # translate STATE
    state_map = translate_state(sas_text)
    df["state"] = df["STATE"].map(state_map)
    df.rename(columns={'YEAR': 'year'}, inplace=True)
    df.drop(columns='STATE', inplace=True)

    # save df as csv
    df.to_csv(csv_output_path, index=False)


parse_dat_to_csv('/home/mrada/Projects/BIOT670I_Capstone_Project/raw/nis/2015/NIS-PUF15.SAS', '/home/mrada/Projects/BIOT670I_Capstone_Project/raw/nis/2015/NISPUF15.DAT', 'test_nis.csv')